# Avance del proyecto — Fase 3
## Núcleo algorítmico, eficiencia e implementación orientada a objetos

**Proyecto:** SIMCE 4° Básico 2025 — Matemática  
**Propósito del notebook:** demostrar, de forma reproducible, la evolución del pipeline validado en F2 hacia una arquitectura modular con POO, recursividad pertinente, pruebas y mediciones de eficiencia.

> **Decisión de arquitectura:** las clases y algoritmos reutilizables **no se definen en las celdas**. Viven en `src/` y este notebook los importa, ejecuta, mide y documenta. Así se evita duplicar implementación y se mantiene una única fuente de verdad.

## 1. Preparación reproducible

La primera celda localiza la raíz del repositorio tanto si el notebook se abre desde `F3/` como desde la raíz. También fija las rutas y muestra las versiones relevantes. No se usan rutas absolutas personales.

In [2]:
from pathlib import Path
import platform
import subprocess
import sys
import pandas as pd
import numpy as np
from IPython.display import display
from pandas.testing import assert_frame_equal

ACTUAL = Path.cwd().resolve()
if (ACTUAL / "src").exists():
    PROJECT_ROOT = ACTUAL
elif (ACTUAL.parent / "src").exists():
    PROJECT_ROOT = ACTUAL.parent
else:
    raise FileNotFoundError(f"No se encontró la raíz del proyecto desde {ACTUAL}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.configuracion import COLUMNAS_SIMCE, encontrar_raw_dir, buscar_simce
from src.carga import cargar_simce, construir_dim_geografia
from src.poo import (
    TransformadorTipos, TransformadorTextos, TransformadorCategorias,
    TransformadorGeografia, TransformadorEfectividad, PipelineSIMCE,
)
from src.algoritmo import (
    aplanar_recursivo, construir_jerarquia_geografica, validar_geografia_recursiva,
    enriquecer_geografia_merge, enriquecer_geografia_diccionario, medir,
)
from src.auditoria import construir_auditoria
from src.transformacion import construir_cobertura_regional
from src.validacion import validar_dataset_final
from src.arquitectura import tabla_arquitectura

RAW_DIR = encontrar_raw_dir()
SIMCE_FILE = buscar_simce(RAW_DIR)
print("Raíz     :", PROJECT_ROOT)
print("Entrada  :", SIMCE_FILE)
print("Python   :", sys.version.split()[0])
print("pandas   :", pd.__version__)
print("NumPy    :", np.__version__)
print("Sistema  :", platform.platform())

Raíz     : C:\Users\crist\f1_s01_evaluacion_entregable_grupo7
Entrada  : C:\Users\crist\f1_s01_evaluacion_entregable_grupo7\data\raw\simce4b2025_rbd_final.csv
Python   : 3.12.6
pandas   : 3.0.5
NumPy    : 2.5.3
Sistema  : Windows-11-10.0.26200-SP0


## 2. Carga y línea base de F2

F3 **no inventa un pipeline nuevo**. Parte del mismo conjunto y de las mismas reglas ya validadas en F2. La dimensión territorial contiene 346 comunas y se valida antes de usarla.

In [3]:
dim_geo = construir_dim_geografia()
df_raw = cargar_simce(SIMCE_FILE, COLUMNAS_SIMCE)

print(f"SIMCE bruto : {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
print(f"DimGeografia: {len(dim_geo):,} comunas")
assert len(dim_geo) == 346
assert not dim_geo.duplicated(["cod_reg_rbd", "cod_pro_rbd", "cod_com_rbd"]).any()
display(df_raw.head(3))

SIMCE bruto : 7,143 filas x 20 columnas
DimGeografia: 346 comunas


,rbd,dvrbd,nom_rbd,cod_reg_rbd,cod_pro_rbd,cod_com_rbd,cod_deprov_rbd,nom_deprov_rbd,cod_depe1,cod_depe2,cod_grupo,cod_rural_rbd,nalu_mate4b_rbd,prom_mate4b_rbd,marca_mate4b_rbd,noaplica,codigo_bbdd,fecha_bbdd,grado,agno
0,7826,3,COLEGIO SAN MIGUEL,10,101,10102,101,Llanquihue,3,2,3.0,1,65,263.0,NaN,0,v22025,20260622,4b,2025
1,11407,3,ESCUELA BASICA LOS OLIVOS,16,162,16206,161,Diguillín,2,1,1.0,2,3,221.0,NaN,0,v22025,20260622,4b,2025
2,10743,3,INSTITUTO SAN FRANCISCO,13,136,13602,135,Talagante,3,2,2.0,1,25,220.0,NaN,0,v22025,20260622,4b,2025


## 3. Pipeline F3: mismo resultado, arquitectura distinta

Cada transformación tiene una responsabilidad concreta. `PipelineSIMCE` compone los pasos y registra el flujo. Las clases concretas reutilizan las funciones de F2 para conservar la lógica de negocio validada.

In [4]:
pipeline = PipelineSIMCE([
    TransformadorTipos(),
    TransformadorTextos(),
    TransformadorCategorias(),
    TransformadorGeografia(dim_geo),
    TransformadorEfectividad(),
])

df_transformado = pipeline.ejecutar(df_raw)
df_final = pipeline.construir_producto_final(df_transformado)
auditoria, resumen_auditoria, resumen_marcas = construir_auditoria(df_transformado)
cobertura = construir_cobertura_regional(df_transformado)
validaciones = validar_dataset_final(df_final, df_transformado, auditoria, cobertura)

print("Pasos:", " -> ".join(pipeline.pasos))
display(pd.DataFrame(pipeline.registro))
print(f"Producto final: {len(df_final):,} filas x {df_final.shape[1]} columnas")
print(f"Excluidos auditados: {len(auditoria):,}")
print("Validaciones F2/F3:", sum(validaciones.values()), "/", len(validaciones))

Pasos: TransformadorTipos -> TransformadorTextos -> TransformadorCategorias -> TransformadorGeografia -> TransformadorEfectividad


,paso,filas_entrada,filas_salida,columnas_salida
0,TransformadorTipos,7143,7143,20
1,TransformadorTextos,7143,7143,20
2,TransformadorCategorias,7143,7143,24
3,TransformadorGeografia,7143,7143,34
4,TransformadorEfectividad,7143,7143,37


Producto final: 6,524 filas x 30 columnas
Excluidos auditados: 619
Validaciones F2/F3: 14 / 14


### Resultado esperado y continuidad

La refactorización debe preservar el resultado: **7.143** registros brutos, **6.524** efectivos y **619** no efectivos auditados. Si estos conteos cambian sin una decisión documentada, F3 habría alterado una regla de negocio y debe investigarse.

In [5]:
assert len(df_raw) == 7143, "Cambió la fuente respecto de la línea base documentada."
assert len(df_final) == 6524, "La refactorización cambió el número de efectivos."
assert len(auditoria) == 619, "La refactorización cambió la auditoría de excluidos."
print("Continuidad F2 -> F3 verificada.")

Continuidad F2 -> F3 verificada.


## 4. POO: evidencia de los principios

- **Herencia:** los transformadores concretos heredan de `Transformador`.
- **Polimorfismo:** el pipeline invoca `transformar(df)` sin preguntar qué clase concreta está ejecutando.
- **Encapsulamiento:** estado interno como `_pasos`, `_registro` y `_dim_geografia` no se modifica directamente desde el notebook.
- **Alta cohesión:** cada clase tiene una responsabilidad.
- **Bajo acoplamiento:** `PipelineSIMCE` depende del contrato `Transformador`, no de una implementación específica.

In [6]:
arquitectura = tabla_arquitectura()
display(arquitectura)
print("Pasos expuestos como estructura inmutable:", type(pipeline.pasos).__name__)
print("Comunas encapsuladas en TransformadorGeografia:", TransformadorGeografia(dim_geo).comunas_dimension)

,clase,hereda_de,responsabilidad,metodos_publicos,modulo
0,Transformador,ABC,Contrato común para transformaciones del pipel...,transformar,src/poo.py
1,TransformadorTipos,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
2,TransformadorTextos,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
3,TransformadorCategorias,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
4,TransformadorGeografia,Transformador,Encapsula la dimensión geográfica necesaria pa...,transformar,src/poo.py
5,TransformadorEfectividad,Transformador,Contrato común para transformaciones del pipel...,transformar,src/poo.py
6,PipelineSIMCE,object,Compone transformadores con bajo acoplamiento ...,"construir_producto_final, ejecutar",src/poo.py


Pasos expuestos como estructura inmutable: tuple
Comunas encapsuladas en TransformadorGeografia: 346


## 5. Flujo y diseño estructurado

El flujo queda explícito y verificable: **carga → tipos → textos → categorías → geografía → efectividad → auditoría/validación → producto final**. El notebook orquesta; `src` implementa. Esta separación permite reutilizar las mismas piezas en otras fases sin copiar celdas.

In [7]:
flujo = pd.DataFrame({
    "orden": range(1, 9),
    "etapa": ["Carga", "Tipos", "Textos", "Categorías", "Geografía", "Efectividad", "Auditoría/validación", "Producto final"],
    "responsable": ["src/carga.py", "src/poo.py", "src/poo.py", "src/poo.py", "src/poo.py", "src/poo.py", "src/auditoria.py + src/validacion.py", "src/transformacion.py"],
})
display(flujo)

,orden,etapa,responsable
0,1,Carga,src/carga.py
1,2,Tipos,src/poo.py
2,3,Textos,src/poo.py
3,4,Categorías,src/poo.py
4,5,Geografía,src/poo.py
5,6,Efectividad,src/poo.py
6,7,Auditoría/validación,src/auditoria.py + src/validacion.py
7,8,Producto final,src/transformacion.py
